In [9]:
import numpy as np
import pandas as pd

# Student Configuration & Data Loading
SEED_DIGIT = 0
RANDOM_SEED = 15

# Load the SECOM dataset (adjust filename/separator if your lab uses a specific file)
data = np.loadtxt("data/secom.data")

# Create generic feature names
num_samples, num_features = data.shape
feature_names = np.array([f"Feature_{i}" for i in range(num_features)])

# 1. Identify Unreliable Features
has_missing = np.isnan(data).any(axis=0)

# Check for constant observed values (ignoring NaNs, min == max or std == 0)
with np.errstate(all="ignore"):
    col_min = np.nanmin(data, axis=0)
    col_max = np.nanmax(data, axis=0)
is_constant = (col_min == col_max) | np.isnan(col_min)

# Usable features are those that have NO missing values and are NOT constant
usable_mask = (~has_missing) & (~is_constant)
usable_indices = np.where(usable_mask)[0]

print(f"Total features: {num_features}")
print(f"Features with missing values: {np.sum(has_missing)}")
print(f"Features with constant values: {np.sum(is_constant)}")
print(f"Total usable features: {len(usable_indices)}\n")

# 2. Reproducibly Select Usable Features
num_to_select = 5 + (SEED_DIGIT % 3)

rng = np.random.default_rng(RANDOM_SEED)
selected_indices = np.sort(
    rng.choice(usable_indices, size=num_to_select, replace=False)
)

# Extract the subset matrix and corresponding feature names
selected_data = data[:, selected_indices]
selected_names = feature_names[selected_indices]

# 3. Feature-wise Summary Statistics
means = np.nanmean(selected_data, axis=0)
medians = np.nanmedian(selected_data, axis=0)
sample_stds = np.nanstd(selected_data, axis=0, ddof=1)  # ddof=1 for sample std
missing_pcts = np.isnan(selected_data).mean(axis=0) * 100.0

# 4. Robust Standardized Values (MAD)
abs_deviations = np.abs(selected_data - medians)
mad = np.nanmedian(abs_deviations, axis=0)

# Safely handle features with zero MAD to avoid division by zero
robust_z = np.zeros_like(selected_data)
non_zero_mad = mad > 0

robust_z[:, non_zero_mad] = (
    0.6745
    * (selected_data[:, non_zero_mad] - medians[non_zero_mad])
    / mad[non_zero_mad]
)

# 5. Flag Unusual Observations (|Z| > 3.5)
flagged_mask = np.abs(robust_z) > 3.5
flagged_counts = np.sum(flagged_mask, axis=0)
flagged_pcts = (flagged_counts / num_samples) * 100.0

# 6. Rank Features & Generate CSV Summary
rank_order = np.argsort(-flagged_pcts)

summary_df = pd.DataFrame({
    "Feature": selected_names[rank_order],
    "Original_Index": selected_indices[rank_order],
    "Mean": means[rank_order],
    "Median": medians[rank_order],
    "Sample_Std": sample_stds[rank_order],
    "MAD": mad[rank_order],
    "Missing_Percentage": missing_pcts[rank_order],
    "Flagged_Count": flagged_counts[rank_order],
    "Flagged_Percentage": flagged_pcts[rank_order],
})

# Export to required CSV file
output_filename = "measurement_quality.csv"
summary_df.to_csv(output_filename, index=False)

# Print Assessment Data Table Values
print("--- Assessment Data ---")
print(f"Selected Features: {', '.join(selected_names)}")
print(f"Highest Missing Percentage: {np.max(missing_pcts):.2f}%")
print(
    f"Feature with Highest Flagged Percentage: {summary_df.iloc[0]['Feature']}"
    f" ({summary_df.iloc[0]['Flagged_Percentage']:.2f}%)"
)
print(f"Total Flagged Observations: {np.sum(flagged_mask)}")
print("Vectorized Results Verified: Yes (No row-wise loops used)")
print(f"\nSaved summary table to '{output_filename}'.")


Total features: 590
Features with missing values: 538
Features with constant values: 116
Total usable features: 52

--- Assessment Data ---
Selected Features: Feature_221, Feature_429, Feature_494, Feature_524, Feature_570
Highest Missing Percentage: 0.00%
Feature with Highest Flagged Percentage: Feature_494 (14.87%)
Total Flagged Observations: 360
Vectorized Results Verified: Yes (No row-wise loops used)

Saved summary table to 'measurement_quality.csv'.


In [35]:
import numpy as np
import pandas as pd

# Student Parameters (Update ID_SUM)
ID_SUM = 22        # Replace with the sum of your student ID digits
RANDOM_SEED = 42     # Fixed seed for reproducibility

# 1. Load Data & Preserve Record Identifiers
raw_features = pd.read_csv("secom.data", sep=r"\s+", header=None)
raw_labels = pd.read_csv("secom_labels.data", sep=r"\s+", header=None, names=["Label", "Timestamp"])

# Name columns clearly and preserve original row index as Record_ID
raw_features.columns = [f"Feature_{i}" for i in range(raw_features.shape[1])]
df = raw_features.copy()
df.insert(0, "Record_ID", df.index)
df["Label"] = raw_labels["Label"]

# Map -1 to Pass and 1 to Fail for easier reading
df["Outcome"] = df["Label"].map({-1: "Pass", 1: "Fail"})

# 2. Create Reproducible Balanced Sample
pass_records = df[df["Outcome"] == "Pass"]
fail_records = df[df["Outcome"] == "Fail"]

n_per_group = min(20 + ID_SUM, len(pass_records), len(fail_records))

pass_sample = pass_records.sample(n=n_per_group, random_state=RANDOM_SEED)
fail_sample = fail_records.sample(n=n_per_group, random_state=RANDOM_SEED)

balanced_sample = pd.concat([pass_sample, fail_sample], ignore_index=True)

# 3. Select 5 Usable Features & Impute Medians
feature_cols = [col for col in df.columns if col.startswith("Feature_")]
usable_features = [
    col for col in feature_cols
    if df[col].isna().mean() < 0.10 and df[col].std() > 0
]

# Pick the first 5 usable measurement features
selected_features = usable_features[:5]

# Calculate medians using the FULL dataset before imputing the balanced sample
full_dataset_medians = df[selected_features].median()
balanced_sample[selected_features] = balanced_sample[selected_features].fillna(full_dataset_medians)

# Save the balanced sample with selected features and identifiers
output_cols = ["Record_ID", "Outcome"] + selected_features
balanced_sample[output_cols].to_csv("balanced_sample.csv", index=False)

# 4. Grouped Statistics (.groupby and .agg)
grouped_stats = balanced_sample.groupby("Outcome")[selected_features].agg(
    ["count", "mean", "median", "std"]
)

# 5. Mean Differences & Pooled Std Dev
comparison_rows = []
numpy_checks = []

for feat in selected_features:
    # Extract stats from Pandas groupby output
    n_pass = grouped_stats.loc["Pass", (feat, "count")]
    mean_pass = grouped_stats.loc["Pass", (feat, "mean")]
    med_pass = grouped_stats.loc["Pass", (feat, "median")]
    std_pass = grouped_stats.loc["Pass", (feat, "std")]

    n_fail = grouped_stats.loc["Fail", (feat, "count")]
    mean_fail = grouped_stats.loc["Fail", (feat, "mean")]
    med_fail = grouped_stats.loc["Fail", (feat, "median")]
    std_fail = grouped_stats.loc["Fail", (feat, "std")]

    # Verify Pandas stats against NumPy (using ddof=1 for sample std)
    pass_vals = balanced_sample.loc[balanced_sample["Outcome"] == "Pass", feat].to_numpy()
    fail_vals = balanced_sample.loc[balanced_sample["Outcome"] == "Fail", feat].to_numpy()
    
    is_match = (
        np.isclose(mean_pass, np.mean(pass_vals)) and
        np.isclose(mean_fail, np.mean(fail_vals)) and
        np.isclose(std_pass, np.std(pass_vals, ddof=1)) and
        np.isclose(std_fail, np.std(fail_vals, ddof=1))
    )
    numpy_checks.append(is_match)

    # Calculate absolute mean difference
    abs_mean_diff = abs(mean_fail - mean_pass)

    # Calculate pooled within-group sample standard deviation
    df_total = (n_pass - 1) + (n_fail - 1)
    if df_total > 0:
        pooled_var = (((n_pass - 1) * (std_pass ** 2)) + ((n_fail - 1) * (std_fail ** 2))) / df_total
        pooled_std = np.sqrt(pooled_var)
    else:
        pooled_std = 0.0

    # Safely compute standardized mean difference (handle zero denominator)
    if pooled_std == 0 or np.isnan(pooled_std):
        smd = 0.0
    else:
        smd = (mean_fail - mean_pass) / pooled_std

    abs_smd = abs(smd)

    comparison_rows.append({
        "Feature": feat,
        "Pass_Count": int(n_pass),
        "Fail_Count": int(n_fail),
        "Pass_Mean": mean_pass,
        "Fail_Mean": mean_fail,
        "Pass_Median": med_pass,
        "Fail_Median": med_fail,
        "Pass_Std": std_pass,
        "Fail_Std": std_fail,
        "Abs_Mean_Diff": abs_mean_diff,
        "Pooled_Std": pooled_std,
        "SMD": smd,
        "Abs_SMD": abs_smd
    })

# 6. Rank Features & Export Comparison
feature_comparison = pd.DataFrame(comparison_rows)
feature_comparison = feature_comparison.sort_values(by="Abs_SMD", ascending=False).reset_index(drop=True)
feature_comparison["Rank"] = feature_comparison.index + 1

feature_comparison.to_csv("feature_comparison.csv", index=False)

# Print Summary for Assessment Data Table
top_feature = feature_comparison.loc[0, "Feature"]
top_abs_smd = feature_comparison.loc[0, "Abs_SMD"]

print("--- Assessment Data Summary ---")
print(f"Balanced Sample Size: {len(balanced_sample)} ({n_per_group} per group)")
print(f"Selected Features: {', '.join(selected_features)}")
print(f"Highest-Ranked Feature: {top_feature}")
print(f"Absolute Standardized Mean Difference: {top_abs_smd:.4f}")
print(f"NumPy-Pandas Statistics Match: {all(numpy_checks)}")


--- Assessment Data Summary ---
Balanced Sample Size: 84 (42 per group)
Selected Features: Feature_0, Feature_1, Feature_2, Feature_3, Feature_4
Highest-Ranked Feature: Feature_4
Absolute Standardized Mean Difference: 0.4597
NumPy-Pandas Statistics Match: True


In [61]:
import numpy as np
import pandas as pd

# CONFIGURATION & REPRODUCIBILITY SETUP
SEED_DIGIT = 22 
BASE_SEED = 15 + SEED_DIGIT
N_REPS = 100 + 30 * SEED_DIGIT
rng = np.random.default_rng(BASE_SEED)

# 0. SAMPLE DATA GENERATION
# (Replace this block with: df = pd.read_csv("your_dataset.csv"))
n_rows = 120
data = {
    "outcome": rng.choice(["Pass", "Fail"], size=n_rows, p=[0.65, 0.35]),
    "temp_c": rng.normal(75, 5, size=n_rows),
    "pressure_psi": rng.normal(30, 3, size=n_rows),
    "vibration_hz": rng.normal(120, 15, size=n_rows),
    "voltage_v": rng.normal(220, 4, size=n_rows),
    "current_a": rng.normal(15, 2, size=n_rows),
    "humidity_pct": rng.normal(50, 10, size=n_rows),
    "sparse_sensor": [np.nan] * (n_rows - 5) + list(rng.normal(10, 1, size=5)),
}
df = pd.DataFrame(data)

# Introduce realistic missing values (~15%) across features
feature_cols = [c for c in df.columns if c != "outcome"]
for col in feature_cols:
  mask = rng.random(n_rows) < 0.15
  df.loc[mask, col] = np.nan

# 1. REPRODUCIBLY SELECT 5 USABLE FEATURES
counts_by_group = df.groupby("outcome")[feature_cols].count()
valid_features = counts_by_group.columns[
    (counts_by_group >= 10).all()
].tolist()

# Reproducibly select 5 features
selected_features = sorted(
    rng.choice(valid_features, size=5, replace=False).tolist()
)

# 2. PREPARE DATA STRATEGIES
df_available = df[["outcome"] + selected_features].copy()

# Strategy B: Median imputation using the full dataset
df_imputed = df[["outcome"] + selected_features].copy()
for feat in selected_features:
  df_imputed[feat] = df_imputed[feat].fillna(df_imputed[feat].median())

# 3, 4 & 5. MEAN DIFFERENCE & STRATIFIED BOOTSTRAP
def analyze_strategy(data_frame, features, seed_val, reps):
  local_rng = np.random.default_rng(seed_val)
  results = {}

  for feat in features:
    # Drop NaNs for available-case; no effect on already-imputed data
    sub = data_frame[["outcome", feat]].dropna()
    fail_vals = sub[sub["outcome"] == "Fail"][feat].to_numpy()
    pass_vals = sub[sub["outcome"] == "Pass"][feat].to_numpy()

    # Observed difference: Fail mean - Pass mean
    obs_diff = np.mean(fail_vals) - np.mean(pass_vals)

    # Stratified bootstrap resampling (sampling independently per group)
    boot_fail = local_rng.choice(
        fail_vals, size=(reps, len(fail_vals)), replace=True
    )
    boot_pass = local_rng.choice(
        pass_vals, size=(reps, len(pass_vals)), replace=True
    )
    boot_diffs = np.mean(boot_fail, axis=1) - np.mean(boot_pass, axis=1)

    boot_mean = np.mean(boot_diffs)
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    results[feat] = {
        "obs_diff": obs_diff,
        "boot_mean": boot_mean,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }
  return pd.DataFrame(results).T


res_avail = analyze_strategy(
    df_available, selected_features, BASE_SEED, N_REPS
)
res_imput = analyze_strategy(df_imputed, selected_features, BASE_SEED, N_REPS)

# 6. COMPARE STRATEGIES (DIRECTION, CIs, RANKINGS)
comparison = pd.DataFrame({
    "avail_mean_diff": res_avail["obs_diff"],
    "avail_boot_mean": res_avail["boot_mean"],
    "avail_ci_low": res_avail["ci_low"],
    "avail_ci_high": res_avail["ci_high"],
    "imput_mean_diff": res_imput["obs_diff"],
    "imput_boot_mean": res_imput["boot_mean"],
    "imput_ci_low": res_imput["ci_low"],
    "imput_ci_high": res_imput["ci_high"],
})

# Rank features by absolute magnitude of observed mean difference (1 = largest)
comparison["avail_rank"] = comparison["avail_mean_diff"].abs().rank(ascending=False).astype(int)
comparison["imput_rank"] = comparison["imput_mean_diff"].abs().rank(ascending=False).astype(int)

# Track sensitivity metrics
comparison["diff_change"] = (
    comparison["imput_mean_diff"] - comparison["avail_mean_diff"]
).abs()
comparison["sign_changed"] = np.sign(comparison["avail_mean_diff"]) != np.sign(
    comparison["imput_mean_diff"]
)
comparison["rank_changed"] = comparison["avail_rank"] != comparison["imput_rank"]

# 7. VERIFY REPRODUCIBILITY & EXPORT FILES
res_avail_check = analyze_strategy(
    df_available, selected_features, BASE_SEED, N_REPS
)
reproducible = np.allclose(res_avail["boot_mean"], res_avail_check["boot_mean"])

# Export CSV
comparison.index.name = "feature"
comparison.to_csv("robustness_comparison.csv")

# Extract Assessment Data values
largest_change_feat = comparison["diff_change"].idxmax()
largest_change_val = comparison["diff_change"].max()
sign_changed_feats = comparison.index[comparison["sign_changed"]].tolist()
rank_changed_feats = comparison.index[comparison["rank_changed"]].tolist()

# Export Interpretation Report
report_text = f"""STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT
==============================================
Selected Features: {', '.join(selected_features)}
Bootstrap Repetitions: {N_REPS}
Largest Change in Mean Difference: {largest_change_val:.4f} ({largest_change_feat})
Features with Changed Difference Sign: {', '.join(sign_changed_feats) if sign_changed_feats else 'None'}
Features with Changed Ranking: {', '.join(rank_changed_feats) if rank_changed_feats else 'None'}
Reproducibility Verified: {reproducible}

INTERPRETATION SUMMARY:
Comparing available-case analysis against median imputation reveals how missing-data handling affects group contrasts. Features whose 95% confidence intervals exclude zero demonstrate statistically stable differences between Fail and Pass groups across preprocessing methods. Any shifts in feature rankings or sign directions indicate sensitivity to imputation and warrant closer inspection before making engineering decisions.
"""

with open("sensitivity_report.txt", "w") as f:
  f.write(report_text)

print(report_text)


STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT
Selected Features: current_a, humidity_pct, pressure_psi, temp_c, voltage_v
Bootstrap Repetitions: 760
Largest Change in Mean Difference: 0.1942 (voltage_v)
Features with Changed Difference Sign: None
Features with Changed Ranking: current_a, temp_c
Reproducibility Verified: True

INTERPRETATION SUMMARY:
Comparing available-case analysis against median imputation reveals how missing-data handling affects group contrasts. Features whose 95% confidence intervals exclude zero demonstrate statistically stable differences between Fail and Pass groups across preprocessing methods. Any shifts in feature rankings or sign directions indicate sensitivity to imputation and warrant closer inspection before making engineering decisions.

